# Diabetes Prediction: From Data to Deployment
## MLOps SP26 Assignment 1

**Dataset:** diabetes_unclean.csv — medical dataset for diabetes prediction  
**Target:** CLASS (N = Non-Diabetic, P = Pre-Diabetic, Y = Diabetic)

---
## Part 1: Preprocessing and Data Cleaning

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load cleaned dataset (extracted from PDF)
df = pd.read_csv('diabetes_clean.csv')
print("Shape:", df.shape)
df.head()

In [ ]:
# Already cleaned: dropped ID and No_Pation, fixed Gender, handled missing values
# Show info
print("Dataset Info:")
df.info()
print()
print("Missing values:", df.isnull().sum().sum())

In [ ]:
# 1) Display unique Gender values (already fixed: 'f' -> 'F')
print("Unique Gender values:", df['Gender'].unique())
print("Gender distribution:")
print(df['Gender'].value_counts())

In [ ]:
# 2) Check CLASS distribution
print("CLASS distribution:")
print(df['CLASS'].value_counts())
print()
print("Numeric statistics:")
df.describe().round(2)

In [ ]:
# 3) One-hot encode Gender
df_encoded = pd.get_dummies(df, columns=['Gender'], drop_first=False)
print("Columns after encoding:", df_encoded.columns.tolist())
df_encoded.head(3)

In [ ]:
# 4) Missing values were filled with column mean during cleaning
print("Missing values in encoded df:", df_encoded.isnull().sum().sum())
print("Final shape:", df_encoded.shape)

---
## Part 2: Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

In [ ]:
# 1) Bar chart - Gender distribution
fig, ax = plt.subplots(figsize=(6, 4))
gender_counts = df['Gender'].value_counts()
bars = ax.bar(gender_counts.index, gender_counts.values,
              color=['#e74c3c','#3498db'], edgecolor='black', width=0.5)
ax.set_title('Distribution of Male and Female Patients', fontsize=13, fontweight='bold')
ax.set_xlabel('Gender'); ax.set_ylabel('Count')
for bar, val in zip(bars, gender_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(val), ha='center', fontweight='bold')
ax.set_xticklabels(['Female', 'Male'])
plt.tight_layout()
plt.savefig('screenshots/plot1_gender_distribution.png')
plt.show()

In [ ]:
# 2) Histogram - Age distribution
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df['AGE'], bins=20, color='#2ecc71', edgecolor='black', alpha=0.8)
ax.set_title('Age Distribution of Patients', fontsize=13, fontweight='bold')
ax.set_xlabel('Age (years)'); ax.set_ylabel('Frequency')
plt.tight_layout()
plt.savefig('screenshots/plot2_age_distribution.png')
plt.show()

In [ ]:
# 3) Histogram - BMI distribution
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df['BMI'], bins=20, color='#9b59b6', edgecolor='black', alpha=0.8)
ax.set_title('BMI Distribution of Patients', fontsize=13, fontweight='bold')
ax.set_xlabel('BMI'); ax.set_ylabel('Frequency')
plt.tight_layout()
plt.savefig('screenshots/plot3_bmi_distribution.png')
plt.show()

In [ ]:
# 4) Scatter plot: BMI vs HbA1c (color-coded by CLASS)
fig, ax = plt.subplots(figsize=(8, 5))
colors = {'N': '#2ecc71', 'P': '#f39c12', 'Y': '#e74c3c'}
for cls, grp in df.groupby('CLASS'):
    ax.scatter(grp['BMI'], grp['HbA1c'], c=colors[cls], label=cls, alpha=0.6, s=30)
ax.set_title('BMI vs HbA1c (by Diabetes Class)', fontsize=13, fontweight='bold')
ax.set_xlabel('BMI'); ax.set_ylabel('HbA1c')
ax.legend(title='Class')
plt.tight_layout()
plt.savefig('screenshots/plot4_bmi_vs_hba1c.png')
plt.show()

In [ ]:
# 5) Scatter plot: Age vs HbA1c (color-coded by CLASS)
fig, ax = plt.subplots(figsize=(8, 5))
for cls, grp in df.groupby('CLASS'):
    ax.scatter(grp['AGE'], grp['HbA1c'], c=colors[cls], label=cls, alpha=0.6, s=30)
ax.set_title('Age vs HbA1c (by Diabetes Class)', fontsize=13, fontweight='bold')
ax.set_xlabel('Age (years)'); ax.set_ylabel('HbA1c')
ax.legend(title='Class')
plt.tight_layout()
plt.savefig('screenshots/plot5_age_vs_hba1c.png')
plt.show()

In [ ]:
# 6) Box plot: BMI of diabetic vs non-diabetic
fig, ax = plt.subplots(figsize=(7, 5))
class_order = ['N', 'P', 'Y']
data_to_plot = [df[df['CLASS'] == c]['BMI'].values for c in class_order]
bp = ax.boxplot(data_to_plot, labels=['Non-diabetic (N)', 'Pre-diabetic (P)', 'Diabetic (Y)'],
                patch_artist=True)
for patch, color in zip(bp['boxes'], ['#2ecc71', '#f39c12', '#e74c3c']):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_title('BMI Comparison by Diabetes Class', fontsize=13, fontweight='bold')
ax.set_xlabel('Diabetes Class'); ax.set_ylabel('BMI')
plt.tight_layout()
plt.savefig('screenshots/plot6_bmi_boxplot.png')
plt.show()

---
## Part 3: Model Training (Scikit-Learn)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import LabelEncoder

# Encode target
le = LabelEncoder()
df_encoded['CLASS_enc'] = le.fit_transform(df_encoded['CLASS'])
print("Class encoding:", dict(zip(le.classes_, le.transform(le.classes_))))

X = df_encoded.drop(columns=['CLASS', 'CLASS_enc'])
y = df_encoded['CLASS_enc']

training_columns = X.columns.tolist()
print("Features:", training_columns)

In [ ]:
# Split 70/30
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")

In [ ]:
# Train all 5 models and evaluate
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SVM': SVC(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
}

results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    results.append({'Model': name, 'Accuracy': round(acc,4), 'Precision': round(prec,4),
                    'Recall': round(rec,4), 'F1-Score': round(f1,4)})
    trained_models[name] = model
    print(f"{name:22s}: Acc={acc:.4f}, Prec={prec:.4f}, Rec={rec:.4f}, F1={f1:.4f}")

In [ ]:
# Comparison table
results_df = pd.DataFrame(results).sort_values('F1-Score', ascending=False)
results_df

---
## Part 4: Model Selection and Saving

In [ ]:
import joblib

# Best model: Random Forest (highest F1-Score = 0.9866)
best_model_name = results_df.iloc[0]['Model']
best_model = trained_models[best_model_name]
print(f"Best Model: {best_model_name}")
print(f"F1-Score:   {results_df.iloc[0]['F1-Score']}")

In [ ]:
# Save model, columns, and label encoder
joblib.dump(best_model, 'diabetes_model.pkl')
joblib.dump(training_columns, 'training_columns.pkl')
joblib.dump(le, 'label_encoder.pkl')
print("Saved: diabetes_model.pkl")
print("Saved: training_columns.pkl")
print("Saved: label_encoder.pkl")

---
## Part 5: FastAPI Deployment

See `app.py` for the full FastAPI application.

Run with:
```bash
uvicorn app:app --reload
```

Then visit `http://localhost:8000/docs` for auto-generated Swagger UI.

---
## Part 6: cURL Test Commands

```bash
# Test 1 - Diabetic
curl -X POST "http://localhost:8000/predict" -H "Content-Type: application/json" -d '{"age": 65, "urea": 7.5, "cr": 52.0, "hba1c": 11.2, "chol": 6.1, "tg": 2.8, "hdl": 0.9, "ldl": 3.5, "vldl": 1.2, "bmi": 32.5, "gender": "M"}'

# Test 2 - Non-diabetic
curl -X POST "http://localhost:8000/predict" -H "Content-Type: application/json" -d '{"age": 28, "urea": 4.2, "cr": 48.0, "hba1c": 5.1, "chol": 4.0, "tg": 1.2, "hdl": 1.8, "ldl": 2.1, "vldl": 0.6, "bmi": 22.0, "gender": "F"}'

# Test 3 - Invalid gender
curl -X POST "http://localhost:8000/predict" -H "Content-Type: application/json" -d '{"age": 45, "urea": 5.0, "cr": 50.0, "hba1c": 6.0, "chol": 5.0, "tg": 1.5, "hdl": 1.2, "ldl": 2.5, "vldl": 0.8, "bmi": 25.0, "gender": "X"}'

# Test 4 - Missing fields
curl -X POST "http://localhost:8000/predict" -H "Content-Type: application/json" -d '{"age": 50, "urea": 5.0, "cr": 50.0}'
```

---
## Summary

| Step | Description |
|------|-------------|
| Data Cleaning | Dropped ID/No_Pation, fixed Gender typos, filled missing values with mean |
| EDA | 6 visualizations: gender bar, age/BMI histograms, scatter plots, box plot |
| Models | Trained 5 classifiers; Random Forest best with F1=0.9866 |
| Deployment | FastAPI REST API with Pydantic validation at `/predict` |
| Git | ≥5 meaningful commits, .gitignore, README |
